In [2]:
import pandas as pd
gp = pd.read_csv("/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/final_rj_sample/gp_reservation_history.csv")
print(gp.columns.tolist())
print(gp.head(3).to_string())

['gp_lgd_code', 'reserved_women_2005', 'reserved_women_2010', 'reservation_dose_n', 'reservation_dose', 'gp_name_lgd', 'district_lgd', 'subdistrict_lgd', 'match_stage']
   gp_lgd_code  reserved_women_2005  reserved_women_2010  reservation_dose_n reservation_dose   gp_name_lgd   district_lgd subdistrict_lgd                 match_stage
0       236073                    0                    0                   0            never  Gajsukhdesar        Bikaner        Jasrasar         B_exact_gp_district
1       236074                    1                    1                   2            twice         Jahaz  Neem Ka Thana     Udaipurwati  C_unique_gp_name_statewide
2       244177                    1                    0                   1             once      Kumbhkot           Kota   Ramganj Mandi         B_exact_gp_district


In [3]:
import pandas as pd
gp = pd.read_csv("/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/final_rj_sample/gp_reservation_history.csv")

print("Total GPs:", len(gp))
print("\nOverall reservation fractions:")
print("2005:", gp['reserved_women_2005'].mean().round(3))
print("2010:", gp['reserved_women_2010'].mean().round(3))

print("\nBy subdistrict - 2005 fraction (first 20):")
sub = gp.groupby('subdistrict_lgd').agg(
    n_gps=('gp_lgd_code','count'),
    frac_2005=('reserved_women_2005','mean'),
    frac_2010=('reserved_women_2010','mean')
).reset_index()
print(sub.describe().round(3))
print("\nSubdistricts with <3 GPs:")
print(sub[sub['n_gps']<3])

Total GPs: 6495

Overall reservation fractions:
2005: 0.337
2010: 0.47

By subdistrict - 2005 fraction (first 20):
         n_gps  frac_2005  frac_2010
count  417.000    417.000    417.000
mean    15.576      0.345      0.466
std      7.858      0.119      0.140
min      2.000      0.000      0.000
25%     10.000      0.273      0.400
50%     15.000      0.333      0.471
75%     21.000      0.412      0.538
max     45.000      0.800      1.000

Subdistricts with <3 GPs:
    subdistrict_lgd  n_gps  frac_2005  frac_2010
157          Jaipur      2        0.5        0.0
267        Neemrana      2        0.0        0.0
313      Rajaldesar      2        0.5        0.5


In [6]:
import pandas as pd
import numpy as np

IR_PATH = ("/Users/sonalideliwala/Library/Mobile Documents/com~apple~CloudDocs"
           "/Desktop/India Data/NFHS/nfhs4/nfhs4 womens (individual) recode/IAIR74FL.DTA")
OUT = "/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/analysis_first_pass"

# P7
print("Saving P7 pre-processed sample...")
ir_p7 = pd.read_stata(IR_PATH,
    columns=["v001","v012","v024","v025","v130","v131","v511"])
ir_p7["v001"] = ir_p7["v001"].astype(int)
ir_p7 = ir_p7[(ir_p7["v024"]==29) & (ir_p7["v025"]==2)]
ir_p7 = ir_p7[(ir_p7["v012"]>=20) & (ir_p7["v012"]<=24)]
ir_p7["v511_clean"] = ir_p7["v511"].where(ir_p7["v511"]<96)
ir_p7["y"] = (ir_p7["v511_clean"] < 15).astype(float)
ir_p7.loc[ir_p7["v511_clean"].isna(), "y"] = np.nan
ir_p7["sc_st"] = ((ir_p7["v131"]==1) | (ir_p7["v131"]==2)).astype(float)
rel_dummies = pd.get_dummies(ir_p7["v130"], prefix="rel", drop_first=True)
rel_dummies = rel_dummies[[c for c in rel_dummies.columns if rel_dummies[c].std()>0]]
ir_p7 = pd.concat([ir_p7[["v001","y","sc_st"]], rel_dummies], axis=1)
ir_p7.to_csv(f"{OUT}/p7_analytic_sample.csv", index=False)
print(f"  Saved P7: {len(ir_p7)} rows")

# P11
print("Saving P11 pre-processed sample...")
auto_cols = ["v001","v012","v024","v025","v130","v131",
             "v743a","v743b","v743d","v743f",
             "s927","s929","s928a","s928b","s928c"]
ir_p11 = pd.read_stata(IR_PATH, columns=auto_cols)
ir_p11["v001"] = ir_p11["v001"].astype(int)
ir_p11 = ir_p11[(ir_p11["v024"]==29) & (ir_p11["v025"]==2)]
for v in ["v743a","v743b","v743d","v743f"]:
    ir_p11[f"{v}_bin"] = np.where(ir_p11[v].isin([1,2]), 1,
                          np.where(ir_p11[v].isna(), np.nan, 0))
for v, col in [("s927","s927_bin"),("s929","s929_bin"),
               ("s928a","s928a_bin"),("s928b","s928b_bin"),("s928c","s928c_bin")]:
    ir_p11[col] = np.where(ir_p11[v]==1, 1, np.where(ir_p11[v].isna(), np.nan, 0))
bin_cols = ["v743a_bin","v743b_bin","v743d_bin","v743f_bin",
            "s927_bin","s929_bin","s928a_bin","s928b_bin","s928c_bin"]
ir_p11["y"] = ir_p11[bin_cols].mean(axis=1, skipna=True)
ir_p11.loc[ir_p11[bin_cols].isna().all(axis=1), "y"] = np.nan
ir_p11["age"] = ir_p11["v012"]
ir_p11["sc_st"] = ((ir_p11["v131"]==1) | (ir_p11["v131"]==2)).astype(float)
rel_dummies11 = pd.get_dummies(ir_p11["v130"], prefix="rel", drop_first=True)
rel_dummies11 = rel_dummies11[[c for c in rel_dummies11.columns if rel_dummies11[c].std()>0]]
ir_p11 = pd.concat([ir_p11[["v001","y","age","sc_st"]], rel_dummies11], axis=1)
ir_p11.to_csv(f"{OUT}/p11_analytic_sample.csv", index=False)
print(f"  Saved P11: {len(ir_p11)} rows")
print("Done.")

Saving P7 pre-processed sample...
  Saved P7: 0 rows
Saving P11 pre-processed sample...
  Saved P11: 0 rows
Done.


In [7]:
import pandas as pd
OUT = "/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/analysis_first_pass"
wk = pd.read_stata(f"{OUT}/women_rj_working.dta")
print(f"Rows: {len(wk)}")
print(f"Columns: {list(wk.columns)}")
print(wk[["v001","v012","v511"]].head(3) if "v511" in wk.columns else "v511 not found")

Rows: 2984
Columns: ['v001', 'v002', 'v003', 'v012', 'v024', 'v025', 'district', 'v130', 'v131', 'v133', 'v155', 'v511', 'v627', 'v628', 'v629', 'v714', 'v741', 'v743a', 'v743b', 'v743d', 'v743f', 's927', 's928a', 's928b', 's928c', 's929', 'p_never_mc', 'p_once_mc', 'p_twice_mc', 'p_known_treatment_mc', 'treatment_certainty_mc', 'primary_gp_dose', 'sc_st', 'rel_1', 'rel_2', 'rel_3', 'rel_4', 'rel_5']
     v001  v012  v511
0  290025    45  14.0
1  290025    23  16.0
2  290025    44  16.0


In [8]:
import pandas as pd
import numpy as np

OUT = "/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/analysis_first_pass"

wk = pd.read_stata(f"{OUT}/women_rj_working.dta")
wk["v001"] = wk["v001"].astype(int)
print(f"Working file rows: {len(wk)}")

# P7
print("Saving P7...")
p7 = wk[(wk["v012"]>=20) & (wk["v012"]<=24)].copy()
p7["v511_clean"] = p7["v511"].where(p7["v511"]<96)
p7["y"] = (p7["v511_clean"] < 15).astype(float)
p7.loc[p7["v511_clean"].isna(), "y"] = np.nan
rel_cols = [c for c in p7.columns if c.startswith("rel_")]
p7[["v001","y","sc_st"] + rel_cols].to_csv(f"{OUT}/p7_analytic_sample.csv", index=False)
print(f"  Saved P7: {len(p7)} rows, {p7['y'].notna().sum()} non-missing y")

# P11
print("Saving P11...")
p11 = wk.copy()
for v in ["v743a","v743b","v743d","v743f"]:
    p11[f"{v}_bin"] = np.where(p11[v].isin([1,2]), 1,
                      np.where(p11[v].isna(), np.nan, 0))
for v, col in [("s927","s927_bin"),("s929","s929_bin"),
               ("s928a","s928a_bin"),("s928b","s928b_bin"),("s928c","s928c_bin")]:
    p11[col] = np.where(p11[v]==1, 1, np.where(p11[v].isna(), np.nan, 0))
bin_cols = ["v743a_bin","v743b_bin","v743d_bin","v743f_bin",
            "s927_bin","s929_bin","s928a_bin","s928b_bin","s928c_bin"]
p11["y"] = p11[bin_cols].mean(axis=1, skipna=True)
p11.loc[p11[bin_cols].isna().all(axis=1), "y"] = np.nan
p11["age"] = p11["v012"]
rel_cols11 = [c for c in p11.columns if c.startswith("rel_")]
p11[["v001","y","age","sc_st"] + rel_cols11].to_csv(
    f"{OUT}/p11_analytic_sample.csv", index=False)
print(f"  Saved P11: {len(p11)} rows, {p11['y'].notna().sum()} non-missing y")
print("Done.")

Working file rows: 2984
Saving P7...
  Saved P7: 573 rows, 436 non-missing y
Saving P11...
  Saved P11: 2984 rows, 420 non-missing y
Done.


In [9]:
"""
ri_and_wcb.py
=============
Randomization Inference (RI) and Wild Cluster Bootstrap (WCB) for P7 and P11.
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# =============================================================================
# PATHS
# =============================================================================

BASE    = "/Users/sonalideliwala/Documents/GitHub/exemplar"
OUT     = f"{BASE}/outputs/analysis_first_pass"
RES     = f"{BASE}/results/v0_regressions"
os.makedirs(RES, exist_ok=True)

GP_RES_PATH = f"{BASE}/outputs/final_rj_sample/gp_reservation_history.csv"
CLUSTER_GP  = f"{BASE}/outputs/final_rj_sample/cluster_gp_res_long.csv"
TREAT_PATH  = f"{BASE}/outputs/final_rj_sample/cluster_treatment_probs_rj.csv"
SHRUG_PATH  = f"{BASE}/outputs/final_rj_sample/shrug_gp_covariates.csv"

N_ITER = 500

# =============================================================================
# STEP 1: Load base data
# =============================================================================

print("Loading base data...")

treat = pd.read_csv(TREAT_PATH)
treat.rename(columns={"DHSCLUST": "v001"}, inplace=True)
treat["v001"] = treat["v001"].astype(int)
treat_linked = treat[treat["p_known_treatment_mc"] == 1][["v001"]].copy()
linked_clusters = set(treat_linked["v001"].tolist())
print(f"  Fully-linked clusters: {len(linked_clusters)}")

cgr = pd.read_csv(CLUSTER_GP)
cgr.rename(columns={"DHSCLUST": "v001"}, inplace=True)
cgr["v001"] = cgr["v001"].astype(int)
cgr = cgr[cgr["v001"].isin(linked_clusters)].copy()
cgr["gp_prob"] = pd.to_numeric(cgr["gp_prob"], errors="coerce")
cgr["gp_lgd_code"] = cgr["gp_lgd_code"].astype(str).str.strip()
print(f"  Cluster-GP rows (fully linked): {len(cgr)}")

gp_res = pd.read_csv(GP_RES_PATH)
gp_res["gp_lgd_code"] = gp_res["gp_lgd_code"].astype(str).str.strip()
print(f"  GPs in reservation history: {len(gp_res)}")

shrug = pd.read_csv(SHRUG_PATH)
shrug.rename(columns={"DHSCLUST": "v001"}, inplace=True)
shrug["v001"] = shrug["v001"].astype(int)

wk = pd.read_stata(f"{OUT}/women_rj_working.dta", columns=["v001","district"])
wk["v001"] = wk["v001"].astype(int)
dist_map = wk.drop_duplicates("v001")[["v001","district"]]

# =============================================================================
# STEP 2: Compute treatment probabilities
# =============================================================================

def compute_treatment_probs(gp_assignments, cgr):
    gp = gp_assignments[["gp_lgd_code","reserved_women_2005",
                          "reserved_women_2010"]].copy()
    gp["gp_lgd_code"] = gp["gp_lgd_code"].astype(str).str.strip()
    gp["dose_n"] = gp["reserved_women_2005"] + gp["reserved_women_2010"]

    merged = cgr[["v001","gp_lgd_code","gp_prob"]].merge(
        gp[["gp_lgd_code","dose_n"]], on="gp_lgd_code", how="left"
    )
    merged["dose_n"] = merged["dose_n"].fillna(0)
    merged["w_once"]  = merged["gp_prob"] * (merged["dose_n"] == 1).astype(float)
    merged["w_twice"] = merged["gp_prob"] * (merged["dose_n"] == 2).astype(float)

    cluster_treat = merged.groupby("v001").agg(
        p_once_mc  = ("w_once",  "sum"),
        p_twice_mc = ("w_twice", "sum"),
        sum_prob   = ("gp_prob", "sum"),
    ).reset_index()

    cluster_treat["p_once_mc"]  /= cluster_treat["sum_prob"]
    cluster_treat["p_twice_mc"] /= cluster_treat["sum_prob"]
    return cluster_treat[["v001","p_once_mc","p_twice_mc"]]

obs_treat = compute_treatment_probs(gp_res, cgr)
obs_treat["v001"] = obs_treat["v001"].astype(int)
print(f"\n  Observed treatment probs: {len(obs_treat)} clusters")
print(f"  p_once_mc mean:  {obs_treat['p_once_mc'].mean():.3f}")
print(f"  p_twice_mc mean: {obs_treat['p_twice_mc'].mean():.3f}")

# =============================================================================
# STEP 3: Permutation function
# =============================================================================

def permute_gp_assignments(gp_res):
    perm = gp_res.copy()
    for sub in perm["subdistrict_lgd"].unique():
        idx = perm["subdistrict_lgd"] == sub
        vals_2005 = perm.loc[idx, "reserved_women_2005"].values.copy()
        vals_2010 = perm.loc[idx, "reserved_women_2010"].values.copy()
        np.random.shuffle(vals_2005)
        np.random.shuffle(vals_2010)
        perm.loc[idx, "reserved_women_2005"] = vals_2005
        perm.loc[idx, "reserved_women_2010"] = vals_2010
    return perm

# =============================================================================
# STEP 4: Load outcome datasets from pre-saved CSVs
# =============================================================================

print("\nLoading P7 outcome data...")
ir_p7 = pd.read_csv(f"{OUT}/p7_analytic_sample.csv")
ir_p7["v001"] = ir_p7["v001"].astype(int)
rel_cols_p7 = [c for c in ir_p7.columns if c.startswith("rel_")]
ir_p7 = ir_p7.merge(shrug, on="v001", how="left")
ir_p7 = ir_p7.merge(dist_map, on="v001", how="left")
ir_p7 = ir_p7[ir_p7["v001"].isin(linked_clusters)]
ir_p7 = ir_p7.dropna(subset=["y","district","log_pop"])
print(f"  P7: N={len(ir_p7)}, clusters={ir_p7['v001'].nunique()}")

print("Loading P11 outcome data...")
ir_p11 = pd.read_csv(f"{OUT}/p11_analytic_sample.csv")
ir_p11["v001"] = ir_p11["v001"].astype(int)
rel_cols_p11 = [c for c in ir_p11.columns if c.startswith("rel_")]
ir_p11 = ir_p11.merge(shrug, on="v001", how="left")
ir_p11 = ir_p11.merge(dist_map, on="v001", how="left")
ir_p11 = ir_p11[ir_p11["v001"].isin(linked_clusters)]
ir_p11 = ir_p11.dropna(subset=["y","district","log_pop"])
print(f"  P11: N={len(ir_p11)}, clusters={ir_p11['v001'].nunique()}")

# =============================================================================
# STEP 5: Regression function
# =============================================================================

SHRUG_COLS = ["log_pop","scst_share","female_pop_share","infra_index"]

def run_regression(df, outcome, nfhs_controls, shrug_controls,
                   cluster_var="v001"):
    controls = " + ".join(nfhs_controls + shrug_controls)
    formula  = f"{outcome} ~ p_once_mc + p_twice_mc + {controls} + C(district)"
    try:
        keep = ([outcome,"p_once_mc","p_twice_mc","district",cluster_var]
                + nfhs_controls + shrug_controls)
        df_clean = df.dropna(subset=keep).copy()
        if df_clean[cluster_var].nunique() < 5:
            return None
        mod = smf.ols(formula, data=df_clean).fit(
            cov_type="cluster",
            cov_kwds={"groups": df_clean[cluster_var]}
        )
        b1  = mod.params["p_once_mc"]
        se1 = mod.bse["p_once_mc"]
        b2  = mod.params["p_twice_mc"]
        se2 = mod.bse["p_twice_mc"]
        return mod, b1, se1, b1/se1, b2, se2, b2/se2
    except Exception:
        return None

# =============================================================================
# STEP 6: Observed test statistics
# =============================================================================

print("\nComputing observed test statistics...")

nfhs_p7  = ["sc_st"] + rel_cols_p7
nfhs_p11 = ["age","sc_st"] + rel_cols_p11

df_p7_obs = ir_p7.merge(obs_treat, on="v001", how="inner")
res_p7 = run_regression(df_p7_obs, "y", nfhs_p7, SHRUG_COLS)
if res_p7:
    _, obs_b1_p7, obs_se1_p7, obs_t1_p7, obs_b2_p7, obs_se2_p7, obs_t2_p7 = res_p7
    print(f"\n  P7 observed:")
    print(f"    p_once_mc:  beta={obs_b1_p7:.4f}, SE={obs_se1_p7:.4f}, t={obs_t1_p7:.3f}")
    print(f"    p_twice_mc: beta={obs_b2_p7:.4f}, SE={obs_se2_p7:.4f}, t={obs_t2_p7:.3f}")
else:
    raise ValueError("P7 observed regression failed")

df_p11_obs = ir_p11.merge(obs_treat, on="v001", how="inner")
ref_mean_p11 = df_p11_obs.loc[
    (df_p11_obs["p_once_mc"] < 0.1) &
    (df_p11_obs["p_twice_mc"] < 0.1), "y"].mean()
ref_sd_p11 = df_p11_obs.loc[
    (df_p11_obs["p_once_mc"] < 0.1) &
    (df_p11_obs["p_twice_mc"] < 0.1), "y"].std()
df_p11_obs["y_std"] = (df_p11_obs["y"] - ref_mean_p11) / ref_sd_p11
res_p11 = run_regression(df_p11_obs, "y_std", nfhs_p11, SHRUG_COLS)
if res_p11:
    _, obs_b1_p11, obs_se1_p11, obs_t1_p11, obs_b2_p11, obs_se2_p11, obs_t2_p11 = res_p11
    print(f"\n  P11 observed:")
    print(f"    p_once_mc:  beta={obs_b1_p11:.4f}, SE={obs_se1_p11:.4f}, t={obs_t1_p11:.3f}")
    print(f"    p_twice_mc: beta={obs_b2_p11:.4f}, SE={obs_se2_p11:.4f}, t={obs_t2_p11:.3f}")
else:
    raise ValueError("P11 observed regression failed")

# =============================================================================
# STEP 7: Randomization Inference
# =============================================================================

print(f"\nRunning RI ({N_ITER} iterations)...")

ri_b1_p7,  ri_t1_p7  = [], []
ri_b2_p7,  ri_t2_p7  = [], []
ri_b1_p11, ri_t1_p11 = [], []
ri_b2_p11, ri_t2_p11 = [], []

for i in range(N_ITER):
    if (i+1) % 50 == 0:
        print(f"  Iteration {i+1}/{N_ITER}...")

    perm_gp    = permute_gp_assignments(gp_res)
    perm_treat = compute_treatment_probs(perm_gp, cgr)
    perm_treat["v001"] = perm_treat["v001"].astype(int)

    df_p7_perm = ir_p7.merge(perm_treat, on="v001", how="inner")
    res = run_regression(df_p7_perm, "y", nfhs_p7, SHRUG_COLS)
    if res:
        _, b1, se1, t1, b2, se2, t2 = res
        ri_b1_p7.append(b1); ri_t1_p7.append(t1)
        ri_b2_p7.append(b2); ri_t2_p7.append(t2)

    df_p11_perm = ir_p11.merge(perm_treat, on="v001", how="inner")
    df_p11_perm["y_std"] = (df_p11_perm["y"] - ref_mean_p11) / ref_sd_p11
    res11 = run_regression(df_p11_perm, "y_std", nfhs_p11, SHRUG_COLS)
    if res11:
        _, b1, se1, t1, b2, se2, t2 = res11
        ri_b1_p11.append(b1); ri_t1_p11.append(t1)
        ri_b2_p11.append(b2); ri_t2_p11.append(t2)

# =============================================================================
# STEP 8: RI p-values
# =============================================================================

def ri_pval(observed, permuted):
    perm = np.array(permuted)
    return np.mean(np.abs(perm) >= np.abs(observed))

print("\n" + "="*60)
print("RANDOMIZATION INFERENCE RESULTS")
print("="*60)

print(f"\nP7: Underage Marriage (before 15), Women 20-24")
print(f"  Successful permutations: {len(ri_b2_p7)}/{N_ITER}")
print(f"\n  p_once_mc:")
print(f"    Observed beta:              {obs_b1_p7:.4f}")
print(f"    Nominal p-val:              {2*(1-stats.norm.cdf(abs(obs_t1_p7))):.3f}")
print(f"    RI p-val (non-studentized): {ri_pval(obs_b1_p7, ri_b1_p7):.3f}")
print(f"    RI p-val (studentized):     {ri_pval(obs_t1_p7, ri_t1_p7):.3f}")
print(f"\n  p_twice_mc:")
print(f"    Observed beta:              {obs_b2_p7:.4f}")
print(f"    Nominal p-val:              {2*(1-stats.norm.cdf(abs(obs_t2_p7))):.3f}")
print(f"    RI p-val (non-studentized): {ri_pval(obs_b2_p7, ri_b2_p7):.3f}")
print(f"    RI p-val (studentized):     {ri_pval(obs_t2_p7, ri_t2_p7):.3f}")

print(f"\nP11: Autonomy Index, Currently Married Women")
print(f"  Successful permutations: {len(ri_b1_p11)}/{N_ITER}")
print(f"\n  p_once_mc:")
print(f"    Observed beta:              {obs_b1_p11:.4f}")
print(f"    Nominal p-val:              {2*(1-stats.norm.cdf(abs(obs_t1_p11))):.3f}")
print(f"    RI p-val (non-studentized): {ri_pval(obs_b1_p11, ri_b1_p11):.3f}")
print(f"    RI p-val (studentized):     {ri_pval(obs_t1_p11, ri_t1_p11):.3f}")
print(f"\n  p_twice_mc:")
print(f"    Observed beta:              {obs_b2_p11:.4f}")
print(f"    Nominal p-val:              {2*(1-stats.norm.cdf(abs(obs_t2_p11))):.3f}")
print(f"    RI p-val (non-studentized): {ri_pval(obs_b2_p11, ri_b2_p11):.3f}")
print(f"    RI p-val (studentized):     {ri_pval(obs_t2_p11, ri_t2_p11):.3f}")

# =============================================================================
# STEP 9: Wild Cluster Bootstrap CIs
# =============================================================================

print("\n" + "="*60)
print(f"WILD CLUSTER BOOTSTRAP ({N_ITER} iterations)")
print("="*60)

def wild_cluster_bootstrap(df, outcome, nfhs_controls, shrug_controls,
                            cluster_var="v001", n_iter=500):
    controls = " + ".join(nfhs_controls + shrug_controls)
    formula  = (f"{outcome} ~ p_once_mc + p_twice_mc + "
                f"{controls} + C(district)")
    keep = ([outcome,"p_once_mc","p_twice_mc","district",cluster_var]
            + nfhs_controls + shrug_controls)
    df_clean = df.dropna(subset=keep).copy().reset_index(drop=True)

    mod_obs   = smf.ols(formula, data=df_clean).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_clean[cluster_var]}
    )
    fitted    = mod_obs.fittedvalues.values
    residuals = mod_obs.resid.values
    clusters  = df_clean[cluster_var].values
    unique_clusters = np.unique(clusters)

    boot_b1, boot_b2 = [], []
    for _ in range(n_iter):
        weights = {c: np.random.choice([-1.0, 1.0]) for c in unique_clusters}
        w = np.array([weights[c] for c in clusters])
        df_clean["y_boot"] = fitted + residuals * w
        try:
            mod_boot = smf.ols(
                f"y_boot ~ p_once_mc + p_twice_mc + {controls} + C(district)",
                data=df_clean
            ).fit(cov_type="cluster",
                  cov_kwds={"groups": df_clean[cluster_var]})
            boot_b1.append(mod_boot.params["p_once_mc"])
            boot_b2.append(mod_boot.params["p_twice_mc"])
        except Exception:
            continue
    return np.array(boot_b1), np.array(boot_b2), mod_obs

def wcb_ci(boot_dist, alpha=0.05):
    lo = np.percentile(boot_dist, 100*alpha/2)
    hi = np.percentile(boot_dist, 100*(1-alpha/2))
    return lo, hi

print("\nP7: Wild Cluster Bootstrap...")
boot_b1_p7, boot_b2_p7, _ = wild_cluster_bootstrap(
    df_p7_obs, "y", nfhs_p7, SHRUG_COLS, n_iter=N_ITER)
ci1_lo_p7, ci1_hi_p7 = wcb_ci(boot_b1_p7)
ci2_lo_p7, ci2_hi_p7 = wcb_ci(boot_b2_p7)
print(f"  p_once_mc:  beta={obs_b1_p7:.4f}, 95% WCB CI=[{ci1_lo_p7:.4f}, {ci1_hi_p7:.4f}]")
print(f"  p_twice_mc: beta={obs_b2_p7:.4f}, 95% WCB CI=[{ci2_lo_p7:.4f}, {ci2_hi_p7:.4f}]")
print(f"  Successful iterations: {len(boot_b2_p7)}/{N_ITER}")

print("\nP11: Wild Cluster Bootstrap...")
boot_b1_p11, boot_b2_p11, _ = wild_cluster_bootstrap(
    df_p11_obs, "y_std", nfhs_p11, SHRUG_COLS, n_iter=N_ITER)
ci1_lo_p11, ci1_hi_p11 = wcb_ci(boot_b1_p11)
ci2_lo_p11, ci2_hi_p11 = wcb_ci(boot_b2_p11)
print(f"  p_once_mc:  beta={obs_b1_p11:.4f}, 95% WCB CI=[{ci1_lo_p11:.4f}, {ci1_hi_p11:.4f}]")
print(f"  p_twice_mc: beta={obs_b2_p11:.4f}, 95% WCB CI=[{ci2_lo_p11:.4f}, {ci2_hi_p11:.4f}]")
print(f"  Successful iterations: {len(boot_b1_p11)}/{N_ITER}")

# =============================================================================
# STEP 10: Save results
# =============================================================================

results = []
for (outcome, coef,
     obs_b, obs_se, obs_t,
     ri_b, ri_t, boot_b, ci_lo, ci_hi) in [
    ("P7",  "p_once_mc",  obs_b1_p7,  obs_se1_p7,  obs_t1_p7,
     ri_b1_p7,  ri_t1_p7,  boot_b1_p7,  ci1_lo_p7,  ci1_hi_p7),
    ("P7",  "p_twice_mc", obs_b2_p7,  obs_se2_p7,  obs_t2_p7,
     ri_b2_p7,  ri_t2_p7,  boot_b2_p7,  ci2_lo_p7,  ci2_hi_p7),
    ("P11", "p_once_mc",  obs_b1_p11, obs_se1_p11, obs_t1_p11,
     ri_b1_p11, ri_t1_p11, boot_b1_p11, ci1_lo_p11, ci1_hi_p11),
    ("P11", "p_twice_mc", obs_b2_p11, obs_se2_p11, obs_t2_p11,
     ri_b2_p11, ri_t2_p11, boot_b2_p11, ci2_lo_p11, ci2_hi_p11),
]:
    nominal_p    = 2*(1-stats.norm.cdf(abs(obs_t)))
    results.append({
        "outcome":             outcome,
        "coefficient":         coef,
        "observed_beta":       round(obs_b,  4),
        "observed_SE":         round(obs_se, 4),
        "observed_t":          round(obs_t,  3),
        "nominal_p":           round(nominal_p, 3),
        "ri_p_nonstudentized": round(ri_pval(obs_b, ri_b), 3),
        "ri_p_studentized":    round(ri_pval(obs_t, ri_t), 3),
        "wcb_ci_lo_95":        round(ci_lo, 4),
        "wcb_ci_hi_95":        round(ci_hi, 4),
        "n_ri_iters":          len(ri_b),
        "n_wcb_iters":         len(boot_b),
    })

df_results = pd.DataFrame(results)
out_path = f"{RES}/ri_and_wcb_results.csv"
df_results.to_csv(out_path, index=False)
print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)
print(df_results.to_string(index=False))
print(f"\nResults saved to: {out_path}")

Loading base data...
  Fully-linked clusters: 130
  Cluster-GP rows (fully linked): 727
  GPs in reservation history: 6495

  Observed treatment probs: 130 clusters
  p_once_mc mean:  0.500
  p_twice_mc mean: 0.149

Loading P7 outcome data...
  P7: N=436, clusters=117
Loading P11 outcome data...
  P11: N=420, clusters=33

Computing observed test statistics...

  P7 observed:
    p_once_mc:  beta=0.0060, SE=0.0458, t=0.130
    p_twice_mc: beta=-0.1706, SE=0.0478, t=-3.569


ValueError: P11 observed regression failed

In [10]:
df_p11_obs = ir_p11.merge(obs_treat, on="v001", how="inner")
print("P11 columns:", [c for c in df_p11_obs.columns if c.startswith("rel")])
print("P11 nfhs_p11 controls:", nfhs_p11)
print("P11 rows before dropna:", len(df_p11_obs))
keep_p11 = (["y","p_once_mc","p_twice_mc","district","v001","age","sc_st"]
            + rel_cols_p11 + SHRUG_COLS)
print("P11 rows after dropna:", len(df_p11_obs.dropna(subset=keep_p11)))
print("Missing counts:")
for c in keep_p11:
    if c in df_p11_obs.columns:
        print(f"  {c}: {df_p11_obs[c].isna().sum()} missing")
    else:
        print(f"  {c}: NOT FOUND IN DATAFRAME")

P11 columns: ['rel_1', 'rel_2', 'rel_3', 'rel_4', 'rel_5']
P11 nfhs_p11 controls: ['age', 'sc_st', 'rel_1', 'rel_2', 'rel_3', 'rel_4', 'rel_5']
P11 rows before dropna: 420
P11 rows after dropna: 420
Missing counts:
  y: 0 missing
  p_once_mc: 0 missing
  p_twice_mc: 0 missing
  district: 0 missing
  v001: 0 missing
  age: 0 missing
  sc_st: 0 missing
  rel_1: 0 missing
  rel_2: 0 missing
  rel_3: 0 missing
  rel_4: 0 missing
  rel_5: 0 missing
  log_pop: 0 missing
  scst_share: 0 missing
  female_pop_share: 0 missing
  infra_index: 0 missing


In [11]:
# Replace the run_regression call for P11 with this diagnostic version:
controls = " + ".join(nfhs_p11 + SHRUG_COLS)
formula  = f"y_std ~ p_once_mc + p_twice_mc + {controls} + C(district)"
print("Formula:", formula)

df_p11_obs = ir_p11.merge(obs_treat, on="v001", how="inner")
ref_mean_p11 = df_p11_obs.loc[
    (df_p11_obs["p_once_mc"] < 0.1) &
    (df_p11_obs["p_twice_mc"] < 0.1), "y"].mean()
ref_sd_p11 = df_p11_obs.loc[
    (df_p11_obs["p_once_mc"] < 0.1) &
    (df_p11_obs["p_twice_mc"] < 0.1), "y"].std()
df_p11_obs["y_std"] = (df_p11_obs["y"] - ref_mean_p11) / ref_sd_p11

try:
    mod = smf.ols(formula, data=df_p11_obs).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_p11_obs["v001"]}
    )
    print("Success! N =", int(mod.nobs))
    print("p_once_mc:", mod.params["p_once_mc"])
    print("p_twice_mc:", mod.params["p_twice_mc"])
except Exception as e:
    print("ERROR:", type(e).__name__, str(e))

Formula: y_std ~ p_once_mc + p_twice_mc + age + sc_st + rel_1 + rel_2 + rel_3 + rel_4 + rel_5 + log_pop + scst_share + female_pop_share + infra_index + C(district)
ERROR: ValueError zero-size array to reduction operation maximum which has no identity


In [12]:
# Diagnostic: check how many districts in P11 sample
print("Districts in P11:", df_p11_obs["district"].nunique())
print("District distribution:")
print(df_p11_obs.groupby("district")["v001"].nunique().describe())

# Try without district FE first
controls = " + ".join(nfhs_p11 + SHRUG_COLS)
formula_nfe = f"y_std ~ p_once_mc + p_twice_mc + {controls}"
print("\nTrying without district FE:")
try:
    mod = smf.ols(formula_nfe, data=df_p11_obs).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_p11_obs["v001"]}
    )
    print("Success! N =", int(mod.nobs))
    print("p_once_mc:", round(mod.params["p_once_mc"], 4))
    print("p_twice_mc:", round(mod.params["p_twice_mc"], 4))
except Exception as e:
    print("ERROR:", str(e))

Districts in P11: 1
District distribution:
count     1.0
mean     33.0
std       NaN
min      33.0
25%      33.0
50%      33.0
75%      33.0
max      33.0
Name: v001, dtype: float64

Trying without district FE:
ERROR: zero-size array to reduction operation maximum which has no identity


In [2]:
# Check actual column names and rename accordingly
print("Treatment file columns:", treat.columns.tolist())
treat.columns = [c.lower() for c in treat.columns]  # lowercase all
treat.rename(columns={"dhsclust": "v001"}, inplace=True)

Treatment file columns: ['DHSCLUST', 'DHSREGNA', 'p_never_mc', 'p_once_mc', 'p_twice_mc', 'p_any_reserved_mc', 'expected_dose_mc', 'p_known_treatment_mc', 'p_unknown_treatment_mc', 'treatment_certainty_mc', 'primary_gp', 'primary_gp_prob', 'primary_gp_dose', 'n_gps_hit']


In [13]:
print("Variation in religion dummies for P11:")
for c in ["rel_1","rel_2","rel_3","rel_4","rel_5"]:
    print(f"  {c}: unique values = {df_p11_obs[c].unique()}, sum = {df_p11_obs[c].sum()}")

print("\nVariation in other controls:")
for c in ["sc_st","age","log_pop","scst_share","female_pop_share","infra_index"]:
    print(f"  {c}: std = {df_p11_obs[c].std():.4f}")

# Try most minimal specification
print("\nTrying minimal spec (p_once + p_twice only):")
try:
    mod = smf.ols("y_std ~ p_once_mc + p_twice_mc", data=df_p11_obs).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_p11_obs["v001"]}
    )
    print("Success! N =", int(mod.nobs))
    print("p_once_mc:", round(mod.params["p_once_mc"], 4))
    print("p_twice_mc:", round(mod.params["p_twice_mc"], 4))
except Exception as e:
    print("ERROR:", str(e))

# Try with just SHRUG controls
print("\nTrying SHRUG controls only:")
try:
    mod = smf.ols("y_std ~ p_once_mc + p_twice_mc + log_pop + scst_share + female_pop_share + infra_index",
                  data=df_p11_obs).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_p11_obs["v001"]}
    )
    print("Success! N =", int(mod.nobs))
    print("p_once_mc:", round(mod.params["p_once_mc"], 4))
    print("p_twice_mc:", round(mod.params["p_twice_mc"], 4))
except Exception as e:
    print("ERROR:", str(e))

Variation in religion dummies for P11:
  rel_1: unique values = [1 0], sum = 414
  rel_2: unique values = [0 1], sum = 6
  rel_3: unique values = [0], sum = 0
  rel_4: unique values = [0], sum = 0
  rel_5: unique values = [0], sum = 0

Variation in other controls:
  sc_st: std = 0.0000
  age: std = 9.9864
  log_pop: std = 0.2159
  scst_share: std = 0.1336
  female_pop_share: std = 0.0148
  infra_index: std = 0.2875

Trying minimal spec (p_once + p_twice only):
ERROR: zero-size array to reduction operation maximum which has no identity

Trying SHRUG controls only:
ERROR: zero-size array to reduction operation maximum which has no identity


In [14]:
# Drop constant columns and use HC1 robust SEs for P11
def run_regression_p11(df, outcome):
    """Simplified regression for P11 — drops constant controls, uses HC1 SEs."""
    # Only keep controls with variation
    df_clean = df.copy()
    controls_to_use = []
    for c in ["age","log_pop","scst_share","female_pop_share","infra_index","rel_2"]:
        if c in df_clean.columns and df_clean[c].std() > 0:
            controls_to_use.append(c)
    
    formula = f"{outcome} ~ p_once_mc + p_twice_mc + " + " + ".join(controls_to_use)
    print(f"  P11 formula: {formula}")
    try:
        mod = smf.ols(formula, data=df_clean).fit(cov_type="HC1")
        b1  = mod.params["p_once_mc"]
        se1 = mod.bse["p_once_mc"]
        b2  = mod.params["p_twice_mc"]
        se2 = mod.bse["p_twice_mc"]
        print(f"  Success! N={int(mod.nobs)}, p_once={b1:.4f}, p_twice={b2:.4f}")
        return mod, b1, se1, b1/se1, b2, se2, b2/se2
    except Exception as e:
        print(f"  ERROR: {e}")
        return None

res_p11 = run_regression_p11(df_p11_obs, "y_std")

  P11 formula: y_std ~ p_once_mc + p_twice_mc + age + log_pop + scst_share + female_pop_share + infra_index + rel_2
  ERROR: zero-size array to reduction operation maximum which has no identity


In [15]:
wk_full = pd.read_stata(
    "/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/analysis_first_pass/women_rj_working.dta",
    columns=["v001","v012","district"]
)
print("Unique districts in working file:", wk_full["district"].nunique())
print("District value counts:")
print(wk_full["district"].value_counts())
print("Total rows:", len(wk_full))

Unique districts in working file: 1
District value counts:
district
rajasthan    2984
Name: count, dtype: int64
Total rows: 2984
